# Stitch → Modular Diffusers — smoke (training-free bounding-box position control)

**Stitch** (arXiv:2509.26644): put *this object* at *this location* on off-the-shelf FLUX — no
training, no detector. Publish PRIVATE `remyxai/stitch-flux-modular` → load via `trust_remote_code` → assert
`StitchBlock` → a cheap 2-object run → the **no-op control**: `regions=None` must be **bit-exact
stock FLUX** → the **cutout-head spike** the brief asks for (dump the text→image attention of the
heads around block 14 and check the default isolates the object). Upload `block.py` first.
Runtime: A100 · `HUGGINGFACE_TOKEN` · accept [FLUX.1-dev](https://huggingface.co/black-forest-labs/FLUX.1-dev).

## 1 · Install + GPU + auth

In [ ]:
!pip install -q "git+https://github.com/huggingface/diffusers.git" transformers accelerate sentencepiece protobuf

In [ ]:
import torch
assert torch.cuda.is_available(); print("GPU:", torch.cuda.get_device_name(0))
from huggingface_hub import login
try:
    from google.colab import userdata; login(userdata.get("HUGGINGFACE_TOKEN"))
except Exception:
    login()
DEV, DT = "cuda", torch.bfloat16

## 2 · Publish PRIVATE (upload `block.py` first)

In [ ]:
import os, json
from huggingface_hub import HfApi
assert os.path.exists("block.py"), "Upload block.py first (the notebook writes the configs beside it)."
open("modular_config.json","w").write(json.dumps({"_class_name": "StitchBlock", "_diffusers_version": "0.41.0.dev0", "auto_map": {"ModularPipelineBlocks": "block.StitchBlock"}}, indent=2))
open("modular_model_index.json","w").write(json.dumps({"_blocks_class_name": "StitchBlock", "_class_name": "ModularPipeline", "_diffusers_version": "0.41.0.dev0", "text_encoder": [null, null, {"pretrained_model_name_or_path": "black-forest-labs/FLUX.1-dev", "revision": null, "subfolder": "text_encoder", "type_hint": ["transformers", "CLIPTextModel"], "variant": null}], "tokenizer": [null, null, {"pretrained_model_name_or_path": "black-forest-labs/FLUX.1-dev", "revision": null, "subfolder": "tokenizer", "type_hint": ["transformers", "CLIPTokenizer"], "variant": null}], "text_encoder_2": [null, null, {"pretrained_model_name_or_path": "black-forest-labs/FLUX.1-dev", "revision": null, "subfolder": "text_encoder_2", "type_hint": ["transformers", "T5EncoderModel"], "variant": null}], "tokenizer_2": [null, null, {"pretrained_model_name_or_path": "black-forest-labs/FLUX.1-dev", "revision": null, "subfolder": "tokenizer_2", "type_hint": ["transformers", "T5TokenizerFast"], "variant": null}], "transformer": [null, null, {"pretrained_model_name_or_path": "black-forest-labs/FLUX.1-dev", "revision": null, "subfolder": "transformer", "type_hint": ["diffusers", "FluxTransformer2DModel"], "variant": null}], "vae": [null, null, {"pretrained_model_name_or_path": "black-forest-labs/FLUX.1-dev", "revision": null, "subfolder": "vae", "type_hint": ["diffusers", "AutoencoderKL"], "variant": null}], "scheduler": [null, null, {"pretrained_model_name_or_path": "black-forest-labs/FLUX.1-dev", "revision": null, "subfolder": "scheduler", "type_hint": ["diffusers", "FlowMatchEulerDiscreteScheduler"], "variant": null}]}, indent=2))
api=HfApi(); REPO="remyxai/stitch-flux-modular"
api.create_repo(REPO, private=True, repo_type="model", exist_ok=True)
for f in ["block.py","modular_config.json","modular_model_index.json"]:
    api.upload_file(path_or_fileobj=f, path_in_repo=f, repo_id=REPO)
print("published PRIVATE:", api.list_repo_files(REPO))

## 3 · Load

In [ ]:
from diffusers import ModularPipeline
from IPython.display import display
pipe = ModularPipeline.from_pretrained("remyxai/stitch-flux-modular", trust_remote_code=True)
assert type(pipe.blocks).__name__ == "StitchBlock", type(pipe.blocks).__name__
print("loaded block:", type(pipe.blocks).__name__)   # expect StitchBlock
pipe.load_components(dtype=DT); pipe.to(DEV)

## 4 · Milestone A — smoke (2 objects, cheap settings)

A low-res few-step 2-object run: the brief's "a red cube to the left of a blue sphere" with
left/right boxes. Must return a 1024² image with no error. (Cheap `region_bind_steps`/steps so the
cell is fast; the e2e uses the paper defaults.)

In [ ]:
import torch
from IPython.display import display
PROMPT = "a red cube to the left of a blue sphere, on a plain studio backdrop"
REGIONS = [
    {"box": [0.05, 0.30, 0.45, 0.75], "prompt": "a red cube"},      # left
    {"box": [0.55, 0.30, 0.95, 0.75], "prompt": "a blue sphere"},   # right
]
g = torch.Generator(DEV).manual_seed(0)
sm = pipe(prompt=PROMPT, regions=REGIONS, height=1024, width=1024,
          region_bind_steps=4, num_inference_steps=12, generator=g).images[0]
assert sm.size == (1024, 1024), sm.size
print("[SMOKE] ran; output size", sm.size)   # expect (1024, 1024)
sm.save("smoke.png"); display(sm.resize((512, 512)))

## 5 · Milestone B — no-op control (`regions=None` == stock FLUX)

The feature-off path installs **no attention processor** and passes no `joint_attention_kwargs`,
so it must reproduce `FluxPipeline` exactly. Compared in **latent** space (the HRDiT/DyPE bit-exact
convention — decoding only adds VAE noise to the comparison), same seed/prompt/steps/guidance.
The modular pipe is parked on CPU while stock runs, so only one FLUX copy sits on the GPU.

In [ ]:
import gc, torch
from diffusers import FluxPipeline

H = W = 512; STEPS = 4
NOPROMPT = "a red barn in a snowy field"

pipe.to("cpu"); gc.collect(); torch.cuda.empty_cache()      # park the modular pipe
stock = FluxPipeline.from_pretrained("black-forest-labs/FLUX.1-dev", torch_dtype=DT).to(DEV)
g2 = torch.Generator(DEV).manual_seed(0)
ref = stock(prompt=NOPROMPT, height=H, width=W, num_inference_steps=STEPS, guidance_scale=3.5,
            max_sequence_length=512, generator=g2, output_type="latent").images.float().cpu()
del stock; gc.collect(); torch.cuda.empty_cache()

pipe.to(DEV)                                               # bring the modular pipe back
g1 = torch.Generator(DEV).manual_seed(0)
ours = pipe(prompt=NOPROMPT, regions=None, height=H, width=W, num_inference_steps=STEPS,
            guidance_scale=3.5, max_sequence_length=512, generator=g1,
            output_type="latent").images[0].float().cpu()

assert ours.shape == ref.shape, (ours.shape, ref.shape)     # both PACKED (1, seq, C)
d = float((ours - ref).abs().max())
print(f"[NO-OP] regions=None vs stock FLUX (packed latents): max|Δ| = {d:.3e}")
print("[NO-OP] PASS — bit-exact, the feature is a true no-op when off"
      if d < 5e-2 else "[NO-OP] REVIEW — inspect the no-op path for stray mutation")

## 6 · Milestone C — cutout-head spike (the one per-build risk)

The brief flags that the cutout head `(block=14, head=20)` is **paper-reported for FLUX.1-dev** and
our build's block/head indexing may differ. Spike it as the brief prescribes: for a **single-object**
prompt, dump the text→image attention of a few heads around block 14 at an early step and score each
by how well its high-attention tokens isolate the object — **in-box share of its attention mass** and
**IoU with the box**. A head that lights up the whole canvas is useless (IoU ≈ box area, mass ≈ 1); a
head that lights up nothing is equally useless. The good head concentrates most of its mass in the
box while covering well under the whole of it.

It reuses the block's **own** seam — the same `StitchProcessor` the pipeline installs, driven by the
same `joint_attention_kwargs['stitch']` payload — so what gets measured is exactly the attention the
cutout would consume. No separate reimplementation to drift.

In [ ]:
import torch
from block import StitchProcessor, _install_attn, _restore_attn, _region_bias
from diffusers.pipelines.flux.pipeline_flux import FluxPipeline

# Resolve the loaded components and the transformer, whichever way this diffusers build exposes them.
comps = getattr(pipe, "components_manager", None) or getattr(pipe, "components", None)
if comps is None:                                   # last resort: the manager attribute itself
    comps = next(v for v in vars(pipe).values()
                 if type(v).__name__ == "ComponentsManager" and hasattr(v, "transformer"))
blk, tr = pipe.blocks, comps.transformer
assert tr is not None, "load_components() must run before the spike"

H = W = 1024
GH = GW = H // 16                                   # packed token grid
OBJ, BOX = "a single red cube", [0.30, 0.25, 0.70, 0.80]
inside = blk._box_tokens(BOX, GH, GW)
emb, pooled, n_real = blk._encode_pass(comps, OBJ, 512, tr.device, tr.dtype)
n_txt = emb.shape[1]
img_ids = FluxPipeline._prepare_latent_image_ids(None, GH, GW, tr.device, tr.dtype)
txt_ids = torch.zeros(n_txt, 3, device=tr.device, dtype=tr.dtype)
latents = torch.randn(1, GH * GW, 64, generator=torch.Generator(tr.device).manual_seed(0),
                      device=tr.device, dtype=tr.dtype)
t = torch.tensor([1.0], device=tr.device)           # an EARLY step, as the brief asks

CANDIDATES = [(b, h) for b in (12, 13, 14, 15, 16) for h in range(20, 24)]
rows = []
for b, h in CANDIDATES:
    if b >= len(tr.single_transformer_blocks) or h >= tr.single_transformer_blocks[0].attn.heads:
        continue
    store = {}
    payload = {"stitch": {"bias": _region_bias(inside, n_txt, GH * GW, torch.float32, tr.device),
                          "capture_id": id(tr.single_transformer_blocks[b].attn), "head": h,
                          "store": store, "n_txt": n_txt}}
    orig = _install_attn(tr, StitchProcessor())
    try:
        with torch.no_grad():
            tr(hidden_states=latents, timestep=t / 1000,
               guidance=torch.full([1], 3.5, device=tr.device), pooled_projections=pooled,
               encoder_hidden_states=emb, txt_ids=txt_ids, img_ids=img_ids,
               joint_attention_kwargs=payload, return_dict=False)
    finally:
        _restore_attn(tr, orig)
    a = store.get("a_txt_img")
    if a is None:
        rows.append(((b, h), None, None)); continue
    w = a[0, :n_real, :].float().mean(dim=0)         # average over non-pad text tokens
    sel = blk._cumulative_foreground(w, 0.95)        # the block's own threshold
    inter, union = int((sel & inside).sum()), int((sel | inside).sum())
    rows.append(((b, h), float(w[inside].sum() / w.sum().clamp_min(1e-9)), inter / max(1, union)))

print(f"{'(block,head)':>12} {'in-box mass':>11} {'IoU(box)':>8}")
for cand, mass, iou in rows:
    tag = "  <- default" if cand == (14, 20) else ""
    print(f"{str(cand):>12} {('%.4f' % mass) if mass is not None else '  n/a':>11} "
          f"{('%.4f' % iou) if iou is not None else ' n/a':>8}{tag}")
good = [c for c, m, i in rows if m is not None and m > 0.6 and 0.1 < i < 0.9]
print("\nheads that isolate the object (in-box mass > 0.6, IoU in (0.1, 0.9)):", good)
print("default (14, 20) among them:", (14, 20) in good)
print("if not, pass cutout_head=(b, h) from that list when you run the pipeline")

## Verdict

`loaded block: StitchBlock` + a 1024² 2-object image + the no-op control matching stock FLUX
(max abs diff ~0) = the two seams (Region-Binding attention mask, cutout capture) work and the
feature is a true no-op when off. Then run `e2e.ipynb` for the position-accuracy claim. If the
probe in milestone C suggests a head other than (14, 20), re-run with `cutout_head=(b, h)` before
filing the result.